# Calculus for Machine Learning Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Numerical derivative from scratch

In [ ]:
```python

def numerical_derivative(f, x, h=1e-7):

    return (f(x + h) - f(x - h)) / (2 * h)

def f(x):

    return x ** 2

for x in [-2, -1, 0, 1, 2]:

    numerical = numerical_derivative(f, x)

    analytical = 2 * x

    print(f"x={x:2d}  f'(x) numerical={numerical:.6f}  analytical={analytical:.1f}")

In [ ]:
```

The numerical derivative matches the analytical one to many decimal places.

### Step 2: Partial derivatives and gradients

In [ ]:
```python

def numerical_gradient(f, point, h=1e-7):

    gradient = []

    for i in range(len(point)):

        point_plus = list(point)

        point_minus = list(point)

        point_plus[i] += h

        point_minus[i] -= h

        partial = (f(point_plus) - f(point_minus)) / (2 * h)

        gradient.append(partial)

    return gradient

def f_multi(point):

    x, y = point

    return x**2 + 3*x*y + y**2

grad = numerical_gradient(f_multi, [1.0, 2.0])

print(f"Numerical gradient at (1,2): {[f'{g:.4f}' for g in grad]}")

print(f"Analytical gradient at (1,2): [2*1+3*2, 3*1+2*2] = [{2*1+3*2}, {3*1+2*2}]")

In [ ]:
```

### Step 3: Gradient descent to find the minimum of f(x) = x^2

In [ ]:
```python

x = 5.0

lr = 0.1

for step in range(20):

    grad = 2 * x

    x = x - lr * grad

    print(f"step {step:2d}  x={x:8.4f}  f(x)={x**2:10.6f}")

In [ ]:
```

Starting at x=5, each step moves closer to x=0 (the minimum).

### Step 4: Gradient descent on a 2D function

In [ ]:
```python

def f_2d(point):

    x, y = point

    return x**2 + y**2

point = [4.0, 3.0]

lr = 0.1

for step in range(30):

    grad = numerical_gradient(f_2d, point)

    point = [p - lr * g for p, g in zip(point, grad)]

    loss = f_2d(point)

    if step % 5 == 0 or step == 29:

        print(f"step {step:2d}  point=({point[0]:7.4f}, {point[1]:7.4f})  f={loss:.6f}")

In [ ]:
```

### Step 5: Comparing numerical and analytical derivatives

In [ ]:
```python

import math

test_functions = [

    ("x^2",      lambda x: x**2,          lambda x: 2*x),

    ("x^3",      lambda x: x**3,          lambda x: 3*x**2),

    ("sin(x)",   lambda x: math.sin(x),   lambda x: math.cos(x)),

    ("e^x",      lambda x: math.exp(x),   lambda x: math.exp(x)),

    ("1/x",      lambda x: 1/x,           lambda x: -1/x**2),

]

x = 2.0

print(f"{'Function':<12} {'Numerical':>12} {'Analytical':>12} {'Error':>12}")

print("-" * 50)

for name, f, df in test_functions:

    num = numerical_derivative(f, x)

    ana = df(x)

    err = abs(num - ana)

    print(f"{name:<12} {num:12.6f} {ana:12.6f} {err:12.2e}")

In [ ]:
```

### Step 6: Computing the Hessian numerically

In [ ]:
```python

def hessian_2d(f, x, y, h=1e-5):

    fxx = (f(x + h, y) - 2 * f(x, y) + f(x - h, y)) / (h ** 2)

    fyy = (f(x, y + h) - 2 * f(x, y) + f(x, y - h)) / (h ** 2)

    fxy = (f(x + h, y + h) - f(x + h, y - h) - f(x - h, y + h) + f(x - h, y - h)) / (4 * h ** 2)

    return [[fxx, fxy], [fxy, fyy]]

def saddle(x, y):

    return x ** 2 - y ** 2

def bowl(x, y):

    return x ** 2 + y ** 2

H_saddle = hessian_2d(saddle, 0.0, 0.0)

H_bowl = hessian_2d(bowl, 0.0, 0.0)

print(f"Saddle Hessian: {H_saddle}")  # [[2, 0], [0, -2]] -- mixed signs

print(f"Bowl Hessian:   {H_bowl}")    # [[2, 0], [0, 2]]  -- both positive

In [ ]:
```

The Hessian of the saddle function has eigenvalues 2 and -2 (mixed signs, confirming a saddle point). The bowl has eigenvalues 2 and 2 (both positive, confirming a minimum).

### Step 7: Taylor approximation in action

In [ ]:
```python

import math

def taylor_approx(f, f_prime, f_double_prime, x0, h, order=2):

    result = f(x0)

    if order >= 1:

        result += f_prime(x0) * h

    if order >= 2:

        result += 0.5 * f_double_prime(x0) * h ** 2

    return result

x0 = 0.0

for h in [0.1, 0.5, 1.0, 2.0]:

    true_val = math.sin(h)

    t1 = taylor_approx(math.sin, math.cos, lambda x: -math.sin(x), x0, h, order=1)

    t2 = taylor_approx(math.sin, math.cos, lambda x: -math.sin(x), x0, h, order=2)

    print(f"h={h:.1f}  sin(h)={true_val:.4f}  order1={t1:.4f}  order2={t2:.4f}")

In [ ]:
```

Near x0=0, sin(x) ~ x (first-order Taylor). The approximation is excellent for small h but breaks down for large h. This is why gradient descent works best with small learning rates -- each step assumes the linear approximation is accurate.

### Step 8: Why this matters for a neural network

In [ ]:
```python

import random

random.seed(42)

w = random.gauss(0, 1)

b = random.gauss(0, 1)

lr = 0.01

xs = [1.0, 2.0, 3.0, 4.0, 5.0]

ys = [3.0, 5.0, 7.0, 9.0, 11.0]

for epoch in range(200):

    total_loss = 0

    dw = 0

    db = 0

    for x, y in zip(xs, ys):

        pred = w * x + b

        error = pred - y

        total_loss += error ** 2

        dw += 2 * error * x

        db += 2 * error

    dw /= len(xs)

    db /= len(xs)

    total_loss /= len(xs)

    w -= lr * dw

    b -= lr * db

    if epoch % 40 == 0 or epoch == 199:

        print(f"epoch {epoch:3d}  w={w:.4f}  b={b:.4f}  loss={total_loss:.6f}")

print(f"\nLearned: y = {w:.2f}x + {b:.2f}")

print(f"Actual:  y = 2x + 1")

In [ ]:
```

Every gradient-based training loop follows this pattern: predict, compute loss, compute gradients, update weights.

## Exercises

In [ ]:
1. Implement `numerical_second_derivative(f, x)` using `numerical_derivative` called twice. Verify that the second derivative of x^3 at x=2 is 12.
2. Use gradient descent to find the minimum of f(x, y) = (x - 3)^2 + (y + 1)^2. Start from (0, 0). The answer should converge to (3, -1).
3. Add momentum to the gradient descent loop: maintain a velocity vector that accumulates past gradients. Compare convergence speed with and without momentum on f(x) = x^4 - 3x^2.